# Calibración del Burstiness Scorer

Este notebook implementa la parte de calibración del **burstiness** para el detector de IA.
El burstiness mide la variabilidad de la perplejidad entre oraciones:
- **Texto humano**: alta variabilidad (burstiness alto)
- **Texto IA**: baja variabilidad (burstiness bajo)

La calibración encuentra el umbral óptimo de burstiness usando percentiles de datos de referencia.

In [ ]:
import numpy as np
import statistics
from typing import List, Dict, Any

## 1. Funciones Auxiliares
Primero definimos las funciones para dividir oraciones y calcular burstiness.

In [ ]:
import re

def split_sentences(text: str) -> List[str]:
    """Divide texto en oraciones usando puntos, signos de interrogación y exclamación."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s for s in sentences if s.strip()]


def calculate_burstiness(perplexities: List[float]) -> float:
    """Calcula burstiness a partir de una lista de perplejidades por oración."""
    if len(perplexities) < 2:
        return 0.0
    mean_ppl = statistics.mean(perplexities)
    std_ppl = statistics.pstdev(perplexities) if len(perplexities) > 1 else 0.0
    return std_ppl / mean_ppl if mean_ppl > 0 else 0.0

## 2. Simulación de Datos
Para probar la calibración, generamos datos sintéticos que simulan:
- Textos humanos: alta variabilidad en perplejidad → burstiness alto
- Textos IA: baja variabilidad en perplejidad → burstiness bajo

In [ ]:
def generate_synthetic_data(n_human: int = 100, n_ai: int = 100) -> Dict[str, Any]:
    """
    Genera datos sintéticos para calibración.
    - Human: perplejidades con alta variabilidad (distribución ancha)
    - AI: perplejidades con baja variabilidad (distribución estrecha)
    """
    np.random.seed(42)
    
    # Textos humanos: alta variabilidad (simulamos 5-10 oraciones por texto)
    human_texts = []
    human_bursts = []
    for _ in range(n_human):
        n_sentences = np.random.randint(5, 10)
        # Perplejidades con alta variabilidad (media ~100, std ~30)
        ppls = np.random.normal(100, 30, n_sentences).tolist()
        ppls = [max(1, p) for p in ppls]  # Evitar valores <= 0
        burst = calculate_burstiness(ppls)
        human_texts.append(ppls)
        human_bursts.append(burst)
    
    # Textos IA: baja variabilidad (simulamos 5-10 oraciones por texto)
    ai_texts = []
    ai_bursts = []
    for _ in range(n_ai):
        n_sentences = np.random.randint(5, 10)
        # Perplejidades con baja variabilidad (media ~50, std ~5)
        ppls = np.random.normal(50, 5, n_sentences).tolist()
        ppls = [max(1, p) for p in ppls]  # Evitar valores <= 0
        burst = calculate_burstiness(ppls)
        ai_texts.append(ppls)
        ai_bursts.append(burst)
    
    return {
        'human_ppls': human_texts,
        'human_bursts': human_bursts,
        'ai_ppls': ai_texts,
        'ai_bursts': ai_bursts
    }

In [ ]:
# Generar datos sintéticos
data = generate_synthetic_data(n_human=200, n_ai=200)
print(f"Datos generados: {len(data['human_bursts'])} textos humanos, {len(data['ai_bursts'])} textos IA")

## 3. Visualización de los Datos
Visualizamos la distribución del burstiness para humanos y IA.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))

# Histograma de burstiness
plt.subplot(1, 2, 1)
sns.histplot(data['human_bursts'], color='blue', alpha=0.6, label='Humano', bins=30)
sns.histplot(data['ai_bursts'], color='red', alpha=0.6, label='IA', bins=30)
plt.title('Distribución de Burstiness')
plt.xlabel('Burstiness')
plt.ylabel('Frecuencia')
plt.legend()

# Boxplot de burstiness
plt.subplot(1, 2, 2)
all_bursts = data['human_bursts'] + data['ai_bursts']
labels = ['Humano'] * len(data['human_bursts']) + ['IA'] * len(data['ai_bursts'])
sns.boxplot(x=labels, y=all_bursts, palette=['blue', 'red'])
plt.title('Boxplot de Burstiness por Tipo')
plt.ylabel('Burstiness')

plt.tight_layout()
plt.show()

## 4. Calibración del Umbral de Burstiness
Esta es la parte clave que se intenta calibrar.

**Lógica:**
- Usamos el percentil 25 de burstiness en humanos (valores bajos de variabilidad humana)
- Usamos el percentil 75 de burstiness en IA (valores altos de variabilidad IA)
- El umbral es el punto medio entre estos dos percentiles

**Interpretación:**
- Si burstiness < umbral → Texto es probablemente IA (baja variabilidad)
- Si burstiness ≥ umbral → Texto es probablemente humano (alta variabilidad)

In [ ]:
def find_burstiness_threshold(
    human_bursts: List[float],
    ai_bursts: List[float]
) -> tuple[float, dict]:
    """
    Encuentra el umbral óptimo de burstiness.
    
    El umbral es el punto medio entre:
    - Percentil 25 de humanos (burstiness bajo en humanos)
    - Percentil 75 de IA (burstiness alto en IA)
    
    Returns:
        umbral, diccionario con estadísticas
    """
    # Calcular percentiles
    burst_human_25 = np.percentile(human_bursts, 25)
    burst_ai_75 = np.percentile(ai_bursts, 75)
    burst_threshold = (burst_human_25 + burst_ai_75) / 2
    
    # Estadísticas adicionales
    stats = {
        'human_p25': burst_human_25,
        'human_p50': np.percentile(human_bursts, 50),
        'human_p75': np.percentile(human_bursts, 75),
        'ai_p25': np.percentile(ai_bursts, 25),
        'ai_p50': np.percentile(ai_bursts, 50),
        'ai_p75': burst_ai_75,
        'burstiness_threshold': burst_threshold,
        'human_mean': np.mean(human_bursts),
        'ai_mean': np.mean(ai_bursts),
        'human_std': np.std(human_bursts),
        'ai_std': np.std(ai_bursts),
    }
    
    return burst_threshold, stats

In [ ]:
# Calcular umbral de burstiness
burst_threshold, stats = find_burstiness_threshold(
    data['human_bursts'], data['ai_bursts']
)

print('=' * 50)
print('CALIBRACIÓN DEL BURSTINESS')
print('=' * 50)

print(' Estadísticas de Burstiness:')
print('-' * 30)
print(f"  Humanos - Media: {stats['human_mean']:.4f}, Std: {stats['human_std']:.4f}")
print(f"  IA      - Media: {stats['ai_mean']:.4f}, Std: {stats['ai_std']:.4f}")

print(' Percentiles:')
print('-' * 30)
print(f"  Humanos - P25: {stats['human_p25']:.4f}, P50: {stats['human_p50']:.4f}, P75: {stats['human_p75']:.4f}")
print(f"  IA      - P25: {stats['ai_p25']:.4f}, P50: {stats['ai_p50']:.4f}, P75: {stats['ai_p75']:.4f}")

print(' Umbral calculado:')
print('-' * 30)
print(f"  UMBRAL BURSTINESS: {burst_threshold:.4f}")
print(f"  (Punto medio entre P25 humanos={stats['human_p25']:.4f} y P75 IA={stats['ai_p75']:.4f})")

print('=' * 50)

## 5. Visualización del Umbral
Graficamos el umbral en relación con las distribuciones.

In [ ]:
plt.figure(figsize=(10, 6))

# Histograma combinado
sns.histplot(data['human_bursts'], color='blue', alpha=0.5, label='Humano', bins=30)
sns.histplot(data['ai_bursts'], color='red', alpha=0.5, label='IA', bins=30)

# Línea del umbral
plt.axvline(burst_threshold, color='black', linestyle='--', linewidth=2, label=f'Umbral: {burst_threshold:.4f}')

# Líneas de percentiles
plt.axvline(stats['human_p25'], color='green', linestyle=':', linewidth=1, label=f"Human P25: {stats['human_p25']:.4f}")
plt.axvline(stats['ai_p75'], color='purple', linestyle=':', linewidth=1, label=f"IA P75: {stats['ai_p75']:.4f}")

plt.title('Calibración del Umbral de Burstiness')
plt.xlabel('Burstiness')
plt.ylabel('Frecuencia')
plt.legend()

plt.grid(True, alpha=0.3)
plt.show()

## 6. Análisis de Clasificación
Evaluamos cómo se comporta el umbral en la clasificación.

In [ ]:
def classify_burstiness(burstiness: float, threshold: float) -> str:
    """Clasifica según el umbral de burstiness."""
    return 'ai' if burstiness < threshold else 'human'


# Aplicar clasificación
human_predictions = [classify_burstiness(b, burst_threshold) for b in data['human_bursts']]
ai_predictions = [classify_burstiness(b, burst_threshold) for b in data['ai_bursts']]

# Métricas
TP = sum(1 for p in ai_predictions if p == 'ai')
FP = sum(1 for p in human_predictions if p == 'ai')
FN = sum(1 for p in ai_predictions if p == 'human')
TN = sum(1 for p in human_predictions if p == 'human')

precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
accuracy = (TP + TN) / (len(data['human_bursts']) + len(data['ai_bursts']))

print(' Métricas de Clasificación (solo con burstiness):')
print('-' * 40)
print(f"  TP: {TP}, FP: {FP}, FN: {FN}, TN: {TN}")
print(f"  Precision:  {precision:.4f}")
print(f"  Recall:     {recall:.4f}")
print(f"  F1-score:   {f1:.4f}")
print(f"  Accuracy:   {accuracy:.4f}")

## 7. Matriz de Confusión
Visualización de la matriz de confusión.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Crear matriz de confusión
y_true = ['human'] * len(data['human_bursts']) + ['ai'] * len(data['ai_bursts'])
y_pred = human_predictions + ai_predictions

cm = confusion_matrix(y_true, y_pred, labels=['human', 'ai'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Human', 'AI'])
disp.plot(cmap='Blues')
plt.title('Matriz de Confusión - Clasificación por Burstiness')
plt.show()

## 8. Función de Calibración Completa
Esta es la función completa de calibración tal como aparece en el código original.

In [ ]:
def find_optimal_thresholds(
    human_texts: List[List[float]],  # Lista de listas de perplejidades por texto
    ai_texts: List[List[float]],      # Lista de listas de perplejidades por texto
    max_samples: int = 100,
) -> tuple[float, dict]:
    """
    Encuentra umbrales óptimos para burstiness.
    
    Esta es la versión exacta de la parte que se intenta calibrar
    en el código original (dual.py: find_optimal_thresholds).
    
    Args:
        human_texts: Lista de perplejidades por oración para cada texto humano
        ai_texts: Lista de perplejidades por oración para cada texto IA
        max_samples: Número máximo de muestras a procesar
    
    Returns:
        burst_threshold: Umbral de burstiness
        stats: Diccionario con estadísticas de calibración
    """
    human_texts = human_texts[:max_samples]
    ai_texts = ai_texts[:max_samples]
    
    # Calcular burstiness para cada texto
    human_bursts = [calculate_burstiness(t) for t in human_texts]
    ai_bursts = [calculate_burstiness(t) for t in ai_texts]
    
    # Burstiness threshold: punto medio entre P25 humanos y P75 IA
    burst_human_25 = np.percentile(human_bursts, 25)
    burst_ai_75 = np.percentile(ai_bursts, 75)
    burst_threshold = (burst_human_25 + burst_ai_75) / 2
    
    stats = {
        'human_bursts_mean': np.mean(human_bursts),
        'ai_bursts_mean': np.mean(ai_bursts),
        'burst_human_25': burst_human_25,
        'burst_ai_75': burst_ai_75,
        'burst_threshold': burst_threshold,
    }
    
    print(f"  Burstiness: {burst_threshold:.4f}")
    
    return burst_threshold, stats

# Probar la función completa
print(' Probando find_optimal_thresholds con datos sintéticos:')
print('-' * 45)
threshold, full_stats = find_optimal_thresholds(data['human_ppls'], data['ai_ppls'])
print(f"Umbral óptimo de burstiness: {threshold:.4f}")

## 9. Resumen

### ¿Qué hemos hecho?
1. **Definido el cálculo de burstiness**: coeficiente de variación (std/mean) de perplejidades por oración
2. **Generado datos sintéticos** que simulan el comportamiento de humanos (alta variabilidad) e IA (baja variabilidad)
3. **Visualizado las distribuciones** para entender la separación entre clases
4. **Calibrado el umbral** usando percentiles: punto medio entre P25 humanos y P75 IA
5. **Evaluado el rendimiento** con métricas estándar de clasificación

### Interpretación del umbral
- **Texto con burstiness < umbral**: Se clasifica como IA (baja variabilidad = artificial)
- **Texto con burstiness ≥ umbral**: Se clasifica como humano (alta variabilidad = natural)

### Aplicación real
En el código original (`dual.py`), esta calibración se usa junto con el umbral de perplejidad
para el enfoque dual: un texto se considera IA solo si cumple **ambos** criterios:
- Perplejidad < umbral_perplejidad
- Burstiness < umbral_burstiness